In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 1. Load dữ liệu đã xử lý
PROCESSED_DIR = '../dataset/processed/'
print("Đang load dữ liệu...")
df = pd.read_parquet(os.path.join(PROCESSED_DIR, 'master_data.parquet'))

print("Đang tính toán phân loại nhu cầu. Quá trình này có thể mất 1-2 phút...")

# MẸO: Chạy thử trên 100 sản phẩm đầu tiên cho nhanh. 
# Khi nào code chạy mượt, bạn COMMENT (thêm dấu #) 2 dòng dưới này lại để chạy cho toàn bộ siêu thị nhé!
sample_items = df['item_id'].unique()[:100]
df_run = df[df['item_id'].isin(sample_items)]
# df_run = df # Bỏ comment dòng này nếu muốn chạy full dữ liệu

def classify_demand(group):
    """Hàm tính ADI, CV2 và phân loại cho từng sản phẩm"""
    demand = group['demand'].values
    
    # Nếu không bán được cái nào trong lịch sử
    if demand.sum() == 0:
        return pd.Series({'ADI': np.nan, 'CV2': np.nan, 'Category': 'No Sales'})
        
    # Tìm index của ngày đầu tiên bắt đầu bán hàng (bỏ qua các ngày zero ban đầu do chưa nhập hàng)
    non_zero_indices = np.where(demand > 0)[0]
    first_sale_idx = non_zero_indices[0]
    active_demand = demand[first_sale_idx:]
    
    # 1. Tính ADI (Số ngày / Số ngày có khách mua)
    days_with_sales = len(active_demand[active_demand > 0])
    if days_with_sales == 0:
        adi = np.nan
    else:
        adi = len(active_demand) / days_with_sales
        
    # 2. Tính CV2 ((Độ lệch chuẩn / Số trung bình)^2)
    mean_demand = np.mean(active_demand)
    if mean_demand == 0:
        cv2 = 0
    else:
        std_demand = np.std(active_demand)
        cv2 = (std_demand / mean_demand) ** 2
        
    # 3. Phân loại
    if adi <= 1.32 and cv2 <= 0.49:
        category = 'Smooth (Đều đặn)'
    elif adi <= 1.32 and cv2 > 0.49:
        category = 'Erratic (Thất thường)'
    elif adi > 1.32 and cv2 <= 0.49:
        category = 'Intermittent (Ngắt quãng)'
    else:
        category = 'Lumpy (Cục bộ)'
        
    return pd.Series({'ADI': round(adi, 2), 'CV2': round(cv2, 2), 'Category': category})

# 2. Chạy Groupby để áp dụng hàm cho từng mặt hàng tại từng cửa hàng
classification_df = df_run.groupby(['item_id', 'store_id']).apply(classify_demand).reset_index()

# Lọc bỏ các món chưa từng bán được
classification_df = classification_df[classification_df['Category'] != 'No Sales']

print("\n✅ Hoàn tất! Thống kê số lượng các nhóm nhu cầu:")
print(classification_df['Category'].value_counts())

print("\nBảng phân loại chi tiết (5 dòng đầu):")
display(classification_df.head(10))

# 3. Lưu kết quả ra file để dùng cho bước Machine Learning
# classification_df.to_csv(os.path.join(PROCESSED_DIR, 'demand_classes.csv'), index=False)